补充一个小技巧

1. 快捷键 `Ctrl+,`（Windows/Linux）/ `Cmd+,`（Mac）打开设置

2. 搜索框输入：`files.exclude`

3. 点击**添加模式**，填入：

    ```
    **/__pycache__
    **/__init__.py
    **/py.typed
    ```

安装插件：Toggle Excluded Files

In [1]:
from langgraph_python.core.create_model import create_model

model = create_model()

# 消息

<img src="./assets/message.svg">

## 消息类型

In [2]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

messages = [
    SystemMessage("你是一个智能天气助手。"),
    HumanMessage("你好！请问今天杭州的天气怎么样？我下午想去西湖散步。"),
    AIMessage(
        content="",
        content_blocks=[{
            "type": "reasoning",
            "reasoning": "我需要调用get_weather工具获取准确的天气数据。"
        }],
        tool_calls=[{
            "id": "call_123456",
            "name": "get_weather",
            "args": {
                "city": "杭州"
            }
        }]
    ),
    ToolMessage(
        content="杭州今天是晴天，气温 25°C",
        tool_call_id="call_123456"  # 必须与AI消息中的tool_calls id匹配
    )
]
await model.ainvoke(messages)

AIMessage(content=[{'signature': '', 'thinking': '思考过程：\n1.  **分析用户的输入**：用户询问今天杭州的天气，并提到下午想去西湖散步。\n2.  **分析工具返回的结果**：工具返回杭州今天是晴天，气温25°C。\n3.  **构建回复**：\n    *   告知用户今天杭州的天气情况（晴天，25°C）。\n    *   结合用户的计划（去西湖散步），给出适当的建议（天气很好，非常适合散步，注意防晒等）。\n    *   保持友好、热情的语气。\n4.  **草拟回复**：\n    “你好！今天杭州的天气非常不错哦！今天是晴天，气温在25°C左右，阳光明媚，非常适合去西湖散步。\n    不过下午紫外线可能比较强，建议你做好防晒措施（比如戴个帽子或涂防晒霜），并记得多喝水。祝你在西湖玩得开心！”\n5.  **检查是否符合角色和约束**：符合“智能天气助手”的角色，准确传达了天气信息，并针对用户意图给出了贴心建议。无需调用其他工具。直接输出回复。', 'type': 'thinking'}, {'text': '你好！今天杭州的天气非常不错哦！\n\n今天是**晴天**，气温在 **25°C** 左右，阳光明媚，气候宜人，非常适合去西湖散步。\n\n不过下午的紫外线可能比较强，建议你去散步时做好防晒措施（比如戴个遮阳帽或涂点防晒霜），并记得带上水杯多补充水分。\n\n祝你在西湖度过一个愉快的下午！如果还有其他问题，随时问我哦。', 'type': 'text'}], additional_kwargs={}, response_metadata={'id': 'msg_9f8d3609-57b0-9004-929f-8ec41c86ca83', 'container': None, 'model': 'qwen3.7-plus', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': None, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 

In [ ]:
# 所有的消息都继承自 BaseMessage
from langchain.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.messages import BaseMessage

print(
    issubclass(SystemMessage, BaseMessage),
    issubclass(HumanMessage, BaseMessage),
    issubclass(AIMessage, BaseMessage),
    issubclass(ToolMessage, BaseMessage),
)

## 消息格式

### 纯文本消息

略

### 图片消息

In [3]:
# 通过 URL 或 base64 编码的图片数据来发送图片消息
url1="https://fastly.picsum.photos/id/57/800/600.jpg?hmac=Geuc5OABDrI_1EI9yvAjSWKQpjo0S6g9Y5mB9JxI4r8"
url2="https://fastly.picsum.photos/id/58/800/600.jpg?hmac=yQHfIMWMXYg4f264WBOGMBWTYbv6NqTJMVz7r67kN3w"
messages = [
    # 一个HumanMessage可以有多个content_block
    HumanMessage([
        {"type": "text", "text": "比较这两张图片风格的差异"},
        {"type": "image", "url": url1},
        {"type": "image", "url": url2}
        # {"type": "image", "base64": "...", "mine_type": "image/png"}  
    ])
]
await model.ainvoke(messages)

CancelledError: 

### 音频消息

注意：需要模型支持ASR能力，否则会报错。本节的示例使用支持ASR的模型。

> 百炼平台没有免费的ASR模型，注意运行期间会产生费用

In [ ]:

import base64
with open("./assets/audio/test.mp3", "rb") as f:
    audio_base64 = base64.b64encode(f.read()).decode()
data_url = f"data:audio/mpeg;base64,{audio_base64}"

asr_model = create_model(
    model_name="qwen3.5-omni-plus",
    provider="openai_reasoning"
)

messages = [
    HumanMessage([
        {"type": "text", "text": "请告诉我这段音频的内容"},
        {"type": "audio", "base64": data_url, "mime_type": "audio/mpeg"}
    ])
]

await asr_model.ainvoke(messages)

### 更多消息格式

其他消息格式需要模型、模型服务商和提供程序的支持

https://docs.langchain.com/oss/python/langchain/messages#content-block-reference

## 消息模板

In [ ]:
from langchain_core.utils.mustache import render

result = render(
    template="Hello {{name}}, you have {{count}} messages", 
    data={"name": "世界", "count": 3}
)

print(result)   # Hello 世界, you have 3 messages

In [4]:
# 使用封装
from langgraph_python.template import prompt_from_filename

prompt_from_filename(
    filename="system",
    data={"cwd":"/jie.chen", "os":"mac", "is_git": True}
)

'你是一个编程助手，你可以：\n\n- 用简洁、准确的语言回答用户问题\n- 调用工具完成任务\n\n# 环境\n\n理解所处的环境，有助于你完成工作:\n - 工作目录（cwd）: `/jie.chen`\n - 操作系统: `mac`\n - 当前工作目录是否处于git仓库管理: `True`\n'